# Building a Rainfall Prediction Classifier

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split,GridSearchCV,StratifiedKFold
from sklearn.metrics import classification_report,confusion_matrix,ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

### Load the data

In [4]:
data = pd.read_csv('weatherAUS_2.csv')
data.sample(5)

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
71575,2012-04-27,Mildura,7.5,20.9,0.0,2.0,9.4,SE,22.0,S,...,80.0,39.0,1024.8,1022.3,3.0,1.0,13.0,20.2,No,No
16087,2011-10-08,Newcastle,14.2,21.2,7.0,NaN,NaN,NaN,NaN,NaN,...,94.0,75.0,NaN,NaN,7.0,4.0,15.5,20.2,Yes,No
116367,2014-01-11,PearceRAAF,19.0,44.5,0.0,NaN,13.1,E,54.0,E,...,27.0,10.0,1015.3,1009.7,NaN,NaN,31.2,42.9,No,No
10726,2013-10-24,CoffsHarbour,21.6,25.0,0.6,6.8,NaN,SSW,50.0,SW,...,50.0,71.0,1012.3,1010.5,7.0,6.0,23.2,24.1,No,No
26158,2014-09-21,Penrith,9.3,21.9,0.0,NaN,NaN,SE,30.0,SSW,...,67.0,38.0,NaN,NaN,NaN,NaN,14.2,21.4,No,No


### Drop all rows with missing values

In [5]:
data.count()

Date             145460
Location         145460
MinTemp          143975
MaxTemp          144199
Rainfall         142199
Evaporation       82670
Sunshine          75625
WindGustDir      135134
WindGustSpeed    135197
WindDir9am       134894
WindDir3pm       141232
WindSpeed9am     143693
WindSpeed3pm     142398
Humidity9am      142806
Humidity3pm      140953
Pressure9am      130395
Pressure3pm      130432
Cloud9am          89572
Cloud3pm          86102
Temp9am          143693
Temp3pm          141851
RainToday        142199
RainTomorrow     142193
dtype: int64

In [6]:
data = data.dropna()
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 56420 entries, 6049 to 142302
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Date           56420 non-null  object 
 1   Location       56420 non-null  object 
 2   MinTemp        56420 non-null  float64
 3   MaxTemp        56420 non-null  float64
 4   Rainfall       56420 non-null  float64
 5   Evaporation    56420 non-null  float64
 6   Sunshine       56420 non-null  float64
 7   WindGustDir    56420 non-null  object 
 8   WindGustSpeed  56420 non-null  float64
 9   WindDir9am     56420 non-null  object 
 10  WindDir3pm     56420 non-null  object 
 11  WindSpeed9am   56420 non-null  float64
 12  WindSpeed3pm   56420 non-null  float64
 13  Humidity9am    56420 non-null  float64
 14  Humidity3pm    56420 non-null  float64
 15  Pressure9am    56420 non-null  float64
 16  Pressure3pm    56420 non-null  float64
 17  Cloud9am       56420 non-null  float64
 18  Cloud3p

### Update the names of rain columns

In [8]:
data = data.rename(columns={
    'RainToday' : 'RainYesterday',
    'RainTomorrow' : 'RainToday'        
                            })

In [ ]:
data['Location'].unique()

array(['Cobar', 'CoffsHarbour', 'Moree', 'NorfolkIsland', 'Sydney',
       'SydneyAirport', 'WaggaWagga', 'Williamtown', 'Canberra', 'Sale',
       'MelbourneAirport', 'Melbourne', 'Mildura', 'Portland', 'Watsonia',
       'Brisbane', 'Cairns', 'Townsville', 'MountGambier', 'Nuriootpa',
       'Woomera', 'PerthAirport', 'Perth', 'Hobart', 'AliceSprings',
       'Darwin'], dtype=object)

### Location selection

In [10]:
data = data[data['Location'].isin(['Melbourne','MelbourneAirport','Watsonia','Sydney'])]
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9247 entries, 31168 to 80997
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Date           9247 non-null   object 
 1   Location       9247 non-null   object 
 2   MinTemp        9247 non-null   float64
 3   MaxTemp        9247 non-null   float64
 4   Rainfall       9247 non-null   float64
 5   Evaporation    9247 non-null   float64
 6   Sunshine       9247 non-null   float64
 7   WindGustDir    9247 non-null   object 
 8   WindGustSpeed  9247 non-null   float64
 9   WindDir9am     9247 non-null   object 
 10  WindDir3pm     9247 non-null   object 
 11  WindSpeed9am   9247 non-null   float64
 12  WindSpeed3pm   9247 non-null   float64
 13  Humidity9am    9247 non-null   float64
 14  Humidity3pm    9247 non-null   float64
 15  Pressure9am    9247 non-null   float64
 16  Pressure3pm    9247 non-null   float64
 17  Cloud9am       9247 non-null   float64
 18  Cloud3pm

### Create a function to map dates to seasons

In [11]:
def date_to_season(date):
    month = int(date.split('-')[1])
    if month in [12, 1, 2]:
        return 'Summer'
    elif month in [3, 4, 5]:
        return 'Autumn'
    elif month in [6, 7, 8]:
        return 'Winter'
    else:
        return 'Spring'

In [13]:
data['Season'] = data['Date'].apply(date_to_season)
data.sample(5)

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainYesterday,RainToday,Season
64646,2010-04-01,MelbourneAirport,12.0,22.4,0.0,2.4,7.7,S,31.0,W,...,49.0,1021.1,1019.5,7.0,6.0,15.7,20.7,No,No,Autumn
65440,2012-07-03,MelbourneAirport,7.6,13.3,0.4,0.4,3.2,SW,28.0,WSW,...,71.0,1024.8,1023.9,7.0,6.0,9.1,12.4,No,No,Winter
32828,2015-08-04,Sydney,8.3,15.1,0.0,3.8,4.6,WSW,33.0,W,...,42.0,1023.0,1018.9,7.0,4.0,8.9,14.4,No,No,Winter
67566,2009-07-02,Melbourne,9.5,13.5,0.2,2.6,0.2,W,52.0,NNW,...,65.0,1005.2,1002.4,7.0,8.0,10.6,12.6,No,Yes,Winter
64727,2010-06-21,MelbourneAirport,3.8,14.3,1.8,2.0,7.5,N,26.0,N,...,52.0,1034.9,1034.4,2.0,4.0,6.7,14.1,Yes,No,Winter


In [14]:
data = data.drop(['Date'],axis=1)
data.sample(5)

,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,...,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainYesterday,RainToday,Season
70385,Melbourne,5.4,16.3,0.0,0.8,4.0,SSW,19.0,NE,SSW,...,70.0,1029.9,1028.8,7.0,4.0,8.9,15.0,No,No,Winter
66444,MelbourneAirport,2.9,11.1,5.4,1.8,6.0,SW,44.0,WSW,SSW,...,58.0,1023.9,1024.6,7.0,6.0,5.2,9.6,Yes,No,Winter
65441,MelbourneAirport,6.1,11.8,1.0,0.8,3.2,S,31.0,SW,S,...,79.0,1030.5,1031.0,3.0,7.0,8.2,10.4,No,No,Winter
32862,Sydney,11.1,24.9,0.2,0.6,9.3,W,54.0,W,W,...,32.0,1018.3,1013.3,2.0,4.0,17.4,22.7,No,No,Spring
67708,Melbourne,18.6,25.9,1.2,6.2,0.0,N,35.0,S,NNW,...,61.0,1009.2,1005.9,8.0,8.0,19.3,24.6,Yes,Yes,Spring
